# Actividad 3 — Patrón Adapter (Estructural)

**Estudiante:** Andrés Felipe Luna Camargo  
**Dominio:** Monitoreo y Gestión de Cuartos Fríos (Cadena de Frío)

---

## 1. Contexto del Problema

En la planta de cuartos fríos tenemos instalados sensores de temperatura de diferentes marcas y generaciones:
- **Sensores Modbus viejos:** Son equipos industriales antiguos que entregan la temperatura en grados Fahrenheit con un método llamado `leer_registro_f()` y su ID es un número hexadecimal.
- **Sensores IoT nuevos:** Son inalámbricos y responden con un diccionario/JSON en grados Kelvin a través de `obtener_telemetria()`.
- **Sensores nativos:** Ya vienen listos y entregan directamente los grados Celsius.

El panel central de monitoreo necesita leer todos los sensores de forma uniforme con dos métodos estándar:
- `leer_celsius() -> float`
- `obtener_id() -> str`

### El problema:
Sin un adaptador, el código del panel de monitoreo tendría que usar `if hasattr(...)` o `isinstance(...)` para ver qué tipo de sensor tiene al frente y hacer las conversiones de unidades (°F a °C y K a °C) ahí mismo. Si mañana compramos sensores de otra marca, toca ensuciar más el código del panel.

**Solución con Adapter:**  
Creamos clases adaptadoras (`AdaptadorModbus`, `AdaptadorIot`) que implementan la interfaz estándar `SensorTarget` y envuelven a los sensores incompatibles para hacer la traducción de forma transparente.


## 2. Código Sin Patrón (Forma Incorrecta)

En esta versión el panel tiene que revisar manualmente qué métodos tiene cada sensor y hacer los cálculos de conversión en medio del ciclo de lectura.


In [1]:
# Código sin patrón: el panel se encarga de convertir y revisar tipos

class SensorModbusViejoDirecto:
    def __init__(self, dir_hex: int, temp_f: float):
        self.dir_hex = dir_hex
        self.temp_f = temp_f

    def leer_registro_f(self) -> float:
        return self.temp_f


class SensorIotDirecto:
    def __init__(self, serial: str, temp_k: float):
        self.serial = serial
        self.temp_k = temp_k

    def obtener_telemetria(self) -> dict:
        return {"serial": self.serial, "temp_kelvin": self.temp_k}


class PanelSinPatron:
    def mostrar_lecturas(self, sensores: list):
        print("=== LECTURAS SIN PATRÓN ===")
        for s in sensores:
            # Problema: revisamos tipos y hacemos fórmulas a mano en el cliente
            if hasattr(s, "leer_registro_f"):
                f = s.leer_registro_f()
                c = round((f - 32) * 5 / 9, 2)
                id_sensor = f"MODBUS-0x{s.dir_hex:02X}"
                print(f"Sensor {id_sensor} -> {c}°C (leído en °F: {f})")
            elif hasattr(s, "obtener_telemetria"):
                datos = s.obtener_telemetria()
                c = round(datos["temp_kelvin"] - 273.15, 2)
                id_sensor = f"IOT-{datos['serial']}"
                print(f"Sensor {id_sensor} -> {c}°C (leído en K: {datos['temp_kelvin']})")
            else:
                print("Sensor no reconocido.")


# Prueba sin patrón
sensores = [
    SensorModbusViejoDirecto(dir_hex=0x12, temp_f=35.6), # ~2°C
    SensorIotDirecto(serial="SN-991", temp_k=203.15),   # -70°C
]

panel = PanelSinPatron()
panel.mostrar_lecturas(sensores)


=== LECTURAS SIN PATRÓN ===
Sensor MODBUS-0x12 -> 2.0°C (leído en °F: 35.6)
Sensor IOT-SN-991 -> -70.0°C (leído en K: 203.15)


## 3. Código Con Patrón Adapter (Forma Correcta)

Estructura de la solución:
- **`SensorTemperaturaTarget` (Target):** Interfaz esperada por el panel (`leer_celsius`, `obtener_id`).
- **`SensorModbusViejo` y `SensorIotApi` (Adaptees):** Clases existentes que no modificamos.
- **`AdaptadorModbus` y `AdaptadorIot` (Adapters):** Envuelven los sensores incompatibles y traducen sus datos a la interfaz Target.
- **`SensorNativoCelsius`:** Sensor nuevo que ya implementa el Target directamente.
- **`PanelMonitoreo` (Cliente):** Trabaja limpiamente con una lista de `SensorTemperaturaTarget`.


In [2]:
from abc import ABC, abstractmethod

# 1. Interfaz que espera nuestro sistema central (Target)
class SensorTemperaturaTarget(ABC):
    @abstractmethod
    def leer_celsius(self) -> float:
        pass

    @abstractmethod
    def obtener_id(self) -> str:
        pass


# 2. Clases incompatibles que ya existen en la planta (Adaptees)
class SensorModbusViejo:
    def __init__(self, dir_hex: int, temp_f: float):
        self.dir_hex = dir_hex
        self.temp_f = temp_f

    def leer_registro_f(self) -> float:
        return self.temp_f


class SensorIotApi:
    def __init__(self, serial: str, temp_k: float):
        self.serial = serial
        self.temp_k = temp_k

    def obtener_telemetria(self) -> dict:
        return {"serial": self.serial, "temp_kelvin": self.temp_k, "bateria": 95}


# 3. Adaptadores que convierten la interfaz de cada sensor al Target
class AdaptadorModbus(SensorTemperaturaTarget):
    def __init__(self, sensor_modbus: SensorModbusViejo):
        self.sensor_modbus = sensor_modbus

    def leer_celsius(self) -> float:
        temp_f = self.sensor_modbus.leer_registro_f()
        temp_c = (temp_f - 32.0) * (5.0 / 9.0)
        return round(temp_c, 2)

    def obtener_id(self) -> str:
        return f"MODBUS-0x{self.sensor_modbus.dir_hex:02X}"


class AdaptadorIot(SensorTemperaturaTarget):
    def __init__(self, sensor_iot: SensorIotApi):
        self.sensor_iot = sensor_iot

    def leer_celsius(self) -> float:
        datos = self.sensor_iot.obtener_telemetria()
        temp_c = datos["temp_kelvin"] - 273.15
        return round(temp_c, 2)

    def obtener_id(self) -> str:
        datos = self.sensor_iot.obtener_telemetria()
        return f"IOT-{datos['serial']}"


# 4. Sensor nativo que ya cumple con la interfaz directamente
class SensorNativoCelsius(SensorTemperaturaTarget):
    def __init__(self, codigo: str, temp_celsius: float):
        self.codigo = codigo
        self.temp_celsius = temp_celsius

    def leer_celsius(self) -> float:
        return round(self.temp_celsius, 2)

    def obtener_id(self) -> str:
        return f"NATIVO-{self.codigo}"


# 5. Cliente: el panel central de monitoreo
class PanelMonitoreo:
    def __init__(self, nombre_planta: str):
        self.nombre_planta = nombre_planta
        self.sensores: list[SensorTemperaturaTarget] = []

    def agregar_sensor(self, sensor: SensorTemperaturaTarget):
        self.sensores.append(sensor)

    def generar_reporte(self, temp_maxima: float):
        print(f"=== Reporte de Monitoreo: {self.nombre_planta} ===")
        print(f"Límite permitido: {temp_maxima}°C")
        print("-" * 55)
        for s in self.sensores:
            temp = s.leer_celsius()
            codigo = s.obtener_id()
            alerta = "ALERTA TEMPERATURA" if temp > temp_maxima else "OK"
            print(f"Sensor: {codigo:<20} | Temp: {temp:>6.2f}°C | Estado: {alerta}")
        print("-" * 55 + "\n")


# 6. Demostración práctica
panel = PanelMonitoreo("Bodega Frigorífica Central")

# Creamos los sensores de distinto tipo
sensor_1 = SensorModbusViejo(dir_hex=0x1A, temp_f=36.5)   # ~2.5°C
sensor_2 = SensorModbusViejo(dir_hex=0x2C, temp_f=48.2)   # ~9.0°C (supera límite)
sensor_3 = SensorIotApi(serial="TEMP-VAC-01", temp_k=203.15) # -70.0°C
sensor_4 = SensorNativoCelsius(codigo="PT100-A", temp_celsius=1.5)

# Los agregamos al panel usando los adaptadores donde hace falta
panel.agregar_sensor(AdaptadorModbus(sensor_1))
panel.agregar_sensor(AdaptadorModbus(sensor_2))
panel.agregar_sensor(AdaptadorIot(sensor_3))
panel.agregar_sensor(sensor_4)

# El panel genera el reporte sin preocuparse por la marca de los sensores
panel.generar_reporte(temp_maxima=4.0)


=== Reporte de Monitoreo: Bodega Frigorífica Central ===
Límite permitido: 4.0°C
-------------------------------------------------------
Sensor: MODBUS-0x1A          | Temp:   2.50°C | Estado: OK
Sensor: MODBUS-0x2C          | Temp:   9.00°C | Estado: ALERTA TEMPERATURA
Sensor: IOT-TEMP-VAC-01      | Temp: -70.00°C | Estado: OK
Sensor: NATIVO-PT100-A       | Temp:   1.50°C | Estado: OK
-------------------------------------------------------



## 4. Diagrama UML

```plantuml
@startuml
skinparam classAttributeIconSize 0

interface SensorTemperaturaTarget {
    + {abstract} leer_celsius() : float
    + {abstract} obtener_id() : str
}

class PanelMonitoreo {
    - sensores: list<SensorTemperaturaTarget>
    + agregar_sensor(sensor: SensorTemperaturaTarget)
    + generar_reporte(temp_maxima: float)
}

PanelMonitoreo o--> "0..*" SensorTemperaturaTarget

class SensorNativoCelsius {
    + leer_celsius() : float
    + obtener_id() : str
}

class AdaptadorModbus {
    - sensor_modbus: SensorModbusViejo
    + leer_celsius() : float
    + obtener_id() : str
}

class AdaptadorIot {
    - sensor_iot: SensorIotApi
    + leer_celsius() : float
    + obtener_id() : str
}

SensorTemperaturaTarget <|.. SensorNativoCelsius
SensorTemperaturaTarget <|.. AdaptadorModbus
SensorTemperaturaTarget <|.. AdaptadorIot

class SensorModbusViejo <<Adaptee>> {
    + dir_hex: int
    + temp_f: float
    + leer_registro_f() : float
}

class SensorIotApi <<Adaptee>> {
    + serial: str
    + temp_k: float
    + obtener_telemetria() : dict
}

AdaptadorModbus o--> SensorModbusViejo
AdaptadorIot o--> SensorIotApi
@enduml
```


## 5. ¿Por qué elegí Adapter y no otro patrón?

Revisando los otros patrones estructurales:

- **¿Por qué no Facade?**  
  Facade sirve cuando uno quiere crear una interfaz simple para manejar una librería o subsistema complejo de muchas clases. Aquí no teníamos un subsistema enredado, solo objetos individuales cuyas funciones tenían nombres y unidades incompatibles.

- **¿Por qué no Decorator?**  
  Decorator se usa para añadirle responsabilidades extra a un objeto sin cambiarle la interfaz. En nuestro caso las clases viejas ya tenían métodos distintos (`leer_registro_f` vs `obtener_telemetria`), así que lo que necesitábamos era traducir la interfaz, no decorarla.

- **¿Por qué no Proxy?**  
  Proxy mantiene la misma interfaz para controlar el acceso (permisos, caché, etc.). Aquí las interfaces eran distintas.

- **Conclusión:**  
  Adapter fue la mejor opción porque nos permitió conectar sensores legados y de terceros al panel central sin tocar el código original de esos sensores ni ensuciar el panel con conversiones manuales.
